# Docker 第1周：容器思维 & 基本操作

> **学习目标**：理解为什么需要 Docker，掌握容器的基本生命周期操作

---

## 开篇：一个真实的故事

想象这个场景：你在本地写了一个 Python Web 应用，一切正常。你很开心，把代码发给同事。

同事说："跑不起来，报错 ImportError。"

你检查了一下——你用的是 Python 3.12，同事用的是 3.9。你的代码用了 `str | None` 类型语法（3.10+ 才支持）。

你帮他升级了 Python。然后又报错："缺少 libpq-dev。"你的机器上装过 PostgreSQL 开发库，同事没装。

折腾了两个小时终于跑起来了。然后部署到服务器——服务器是 CentOS，glibc 版本不对……

这是经典的 **"在我机器上能跑"**（Works on my machine）问题。Docker 就是为解决这个问题而生的。

---

## Docker 是什么？一句话版本

**Docker 把你的应用连同它需要的一切（代码、运行时、系统库、环境变量）打包成一个标准化的"集装箱"，这个集装箱可以在任何安装了 Docker 的机器上原封不动地运行。**

类比：
- **传统部署** = 搬家时把家具一件一件搬上车，到了新家再重新摆放
- **Docker** = 把整个房间打包进一个集装箱，卡车运到哪，打开门，一切原样

---

## 虚拟机 vs 容器

很多人第一次听到 Docker 会问："这和虚拟机有什么区别？"

```
虚拟机架构：                        容器架构：
┌──────────┬──────────┐             ┌──────────┬──────────┐
│  App A   │  App B   │             │  App A   │  App B   │
├──────────┼──────────┤             ├──────────┼──────────┤
│ Guest OS │ Guest OS │             │ 容器引擎 (共享内核)   │
├──────────┴──────────┤             ├─────────────────────┤
│   Hypervisor        │             │   Host OS (Linux)   │
├─────────────────────┤             ├─────────────────────┤
│   Host OS           │             │   硬件               │
├─────────────────────┤             └─────────────────────┘
│   硬件              │
└─────────────────────┘
```

**关键区别**：
- 虚拟机：每个 VM 有自己的完整操作系统（Guest OS），**重**（几 GB）、**慢**（分钟级启动）
- 容器：所有容器**共享宿主机的操作系统内核**，**轻**（几十 MB）、**快**（秒级启动）

打个比方：
- 虚拟机 = 在小区里给每户人家单独盖一栋楼（地基、水管、电线全独立）
- 容器 = 在同一栋楼里给每户人家隔出独立房间（共享地基和水电总管，但彼此隔离）

---

## Docker 的核心架构

Docker 是 Client-Server 架构：

```
你敲命令(docker run ...)
    ↓
Docker CLI (客户端)
    ↓ 通过 Unix Socket / REST API 通信
Docker Daemon ( dockerd，后台服务)
    ↓ 管理
容器、镜像、网络、存储
```

当你执行 `docker run nginx` 时发生了什么：
1. CLI 把请求发给 daemon
2. Daemon 检查本地有没有 nginx 镜像 → 没有就去 Docker Hub 拉取
3. Daemon 用镜像创建容器，分配网络、存储
4. 容器启动，nginx 进程开始运行

---

## 安装 Docker

> **注意**：以下命令在终端执行，不在 Python/Jupyter 中执行。这里用 `!` 前缀只是为了在 notebook 中演示。

### Linux (Ubuntu/Debian)

```bash
# 官方推荐方式：用 Docker 官方脚本
curl -fsSL https://get.docker.com | sudo sh

# 把当前用户加入 docker 组（免得每次 sudo）
sudo usermod -aG docker $USER
# 注销重新登录后生效
```

### macOS
- 下载 Docker Desktop：https://www.docker.com/products/docker-desktop/
- 安装 .dmg 文件，拖到 Applications

### 验证安装

In [ ]:
# 在 Jupyter 中运行 shell 命令用 ! 前缀
! docker --version
! docker run hello-world

`hello-world` 是最简单的 Docker 镜像，它只做一件事：打印一段欢迎信息然后退出。
如果这条命令成功，说明 Docker 安装正确。

---

## 镜像 vs 容器：两个核心概念

这是 Docker 世界里最重要的两个概念，必须彻底分清：

| 概念 | 类比 | 特点 |
|------|------|------|
| **镜像（Image）** | 光盘 / 安装包 / 类（Class） | 只读、不可变 |
| **容器（Container）** | 运行中的程序 / 实例（Object） | 可写、有生命周期 |

更形象的比喻：
- **镜像** = 饼干的模具。模具本身不能吃，但你可以用它做出很多饼干
- **容器** = 用模具做出的饼干。每个饼干是独立的，你咬了一口饼干 A，不影响饼干 B

**同一个镜像可以启动无数个容器，每个容器互相隔离，互不干扰。**

### 镜像的分层存储

镜像不是一个大文件，而是由多个**只读层（layer）**叠加而成：

```
┌─────────────────┐
│   你的应用代码    │ ← 第3层（你写的 COPY . .）
├─────────────────┤
│   pip install    │ ← 第2层（RUN pip install -r requirements.txt）
├─────────────────┤
│  Python 3.12     │ ← 第1层（FROM python:3.12-slim）
├─────────────────┤
│  Debian 基础层    │ ← 第0层（slim 镜像的底层 OS）
└─────────────────┘
```

每一层只存储和上一层的差异（diff），这叫做 **UnionFS（联合文件系统）**。

好处：
- **节省空间**：10 个 Python 应用共享同一个 `python:3.12-slim` 层，只存一份
- **加速构建**：改了代码只重建最上面一层，下面的层用缓存
- **加速分发**：拉取镜像时，已有的层不用重复下载

---

## 动手：拉取和运行第一个容器

In [ ]:
# 拉取 nginx 镜像（Web 服务器）
! docker pull nginx

# 查看本地已有的镜像
! docker images

In [ ]:
# 后台运行 nginx 容器
# -d: 后台运行（detached mode）
# --name: 给容器起个名字（不指定则随机生成）
# -p 8080:80: 把宿主机的 8080 端口映射到容器的 80 端口
! docker run -d --name my-nginx -p 8080:80 nginx

# 查看正在运行的容器
! docker ps

In [ ]:
# 用 curl 验证 nginx 是否正常响应
! curl -s http://localhost:8080 | head -5

如果看到 HTML 内容返回，说明容器运行正常。

你也可以在浏览器打开 `http://localhost:8080`，会看到 nginx 的默认欢迎页。

---

## 容器的生命周期管理

一个容器从生到死经历的完整流程：

```
创建(create) → 启动(start) → 运行(running) → 暂停(pause)/停止(stop) → 删除(rm)
```

### 核心命令速查

| 命令 | 作用 |
|------|------|
| `docker ps` | 查看运行中的容器 |
| `docker ps -a` | 查看所有容器（包括已停止的） |
| `docker start <name>` | 启动已停止的容器 |
| `docker stop <name>` | 优雅停止（发 SIGTERM，等 10 秒再 SIGKILL） |
| `docker restart <name>` | 重启容器 |
| `docker rm <name>` | 删除容器（要先 stop） |
| `docker rm -f <name>` | 强制删除（不管容器是否在运行） |
| `docker logs <name>` | 查看容器日志 |
| `docker exec -it <name> bash` | 进入容器内部 |
| `docker inspect <name>` | 查看容器详细信息（JSON 格式） |

In [ ]:
# 查看所有容器（包括已停止的）
! docker ps -a

In [ ]:
# 查看 nginx 容器的日志
! docker logs my-nginx

# 跟踪日志（实时）
# ! docker logs -f my-nginx

In [ ]:
# 进入容器内部执行命令
! docker exec my-nginx ls /usr/share/nginx/html

# 进入容器的交互式 shell
# docker exec -it my-nginx bash
# 进去后可以随意探索，exit 退出

In [ ]:
# 停止并删除实验容器
! docker stop my-nginx
! docker rm my-nginx

# 验证已删除
! docker ps -a | grep my-nginx || echo "已删除"

---

## 端口映射：让外部能访问容器

容器有自己的网络命名空间，默认和宿主机网络隔离。要把容器的端口"暴露"出去，用 `-p` 参数。

```
docker run -p <宿主机端口>:<容器端口> <镜像>
```

```
你的浏览器                    Docker 容器
localhost:8080  ──────────▶  nginx:80
     ↑                            ↑
  宿主机端口                   容器端口
  (对外)                       (对内)
```

In [ ]:
# 端口映射演示：启动一个 Python HTTP 服务器
! docker run -d --name py-server -p 9000:8000 python:3.12-slim \
  python -c "from http.server import HTTPServer, SimpleHTTPRequestHandler; \
  HTTPServer(('0.0.0.0', 8000), SimpleHTTPRequestHandler).serve_forever()"

# 验证：访问宿主机的 9000 端口，流量会被转发到容器的 8000 端口
! curl -s -o /dev/null -w "%{http_code}" http://localhost:9000

In [ ]:
# 清理
! docker rm -f py-server

---

## 卷挂载：让容器能访问宿主机的文件

容器默认的文件系统是**临时的**——容器删除后，里面的数据也丢了。

要想数据持久化，或者让容器读取宿主机上的文件，用 `-v` 挂载：

```
docker run -v <宿主机路径>:<容器路径> <镜像>
```

In [ ]:
# 创建一个测试目录和文件
! mkdir -p /tmp/docker-test
! echo "<h1>来自宿主机的问候</h1>" > /tmp/docker-test/index.html

# 挂载到 nginx 的默认网页目录
! docker run -d --name nginx-with-volume -p 8081:80 \
  -v /tmp/docker-test:/usr/share/nginx/html nginx

# 验证：nginx 显示的是我们自己的 HTML
! curl -s http://localhost:8081

In [ ]:
# 在宿主机修改文件，容器内立即生效
! echo "<h1>文件已更新！</h1>" > /tmp/docker-test/index.html
! curl -s http://localhost:8081

# 这就是开发时最常用的 bind mount：代码在宿主机，运行时在容器

In [ ]:
# 清理
! docker rm -f nginx-with-volume

---

## 环境变量：`-e` 传配置

很多镜像通过环境变量来配置行为。比如 MySQL 镜像需要 `MYSQL_ROOT_PASSWORD`。

```bash
docker run -e MYSQL_ROOT_PASSWORD=secret -d mysql:8
```

多条环境变量：
```bash
docker run -e KEY1=val1 -e KEY2=val2 ...
# 或从文件加载
docker run --env-file .env ...
```

In [ ]:
# 用 Python 容器演示环境变量
! docker run --rm python:3.12-slim python -c "
import os
print('HOME:', os.environ.get('HOME'))
print('MY_VAR:', os.environ.get('MY_VAR', '未设置'))
"

print('---')

# 传入自定义环境变量
! docker run --rm -e MY_VAR=hello_docker python:3.12-slim python -c "
import os
print('MY_VAR:', os.environ.get('MY_VAR', '未设置'))
"

# --rm 表示容器退出后自动删除，适合一次性任务

---

## 容器网络：让容器之间通信

Docker 默认创建 `bridge` 网络。同一网络下的容器可以用**容器名作为 DNS** 互相访问。

In [ ]:
# 创建一个自定义网络
! docker network create my-net

# 启动 Redis 容器，加入自定义网络
! docker run -d --name my-redis --network my-net redis:7-alpine

# 启动 Python 容器（同一网络），测试连接
! docker run --rm --network my-net python:3.12-slim pip install redis -q && \
  python -c "
import socket
# 用容器名解析 IP
ip = socket.gethostbyname('my-redis')
print(f'my-redis 的 IP: {ip}')
print('容器名可以直接当 DNS 使用！')
"

In [ ]:
# 查看网络信息
! docker network inspect my-net | python3 -c "import sys,json; d=json.load(sys.stdin)[0]; [print(f'{c[\"Name\"]}: {c[\"IPv4Address\"]}') for c in d['Containers'].values()]" 2>/dev/null || docker network inspect my-net

# 清理
! docker rm -f my-redis
! docker network rm my-net

---

## 镜像管理常用命令

In [ ]:
# 查看本地所有镜像
! docker images

# 查看镜像详细信息
# ! docker inspect nginx

# 查看镜像的历史层
! docker history nginx | head -10

In [ ]:
# 给镜像打标签
! docker tag nginx my-nginx:v1
! docker images | grep my-nginx

# 注意：打 tag 不会复制镜像，只是一个别名（指针）

# 删除标签
! docker rmi my-nginx:v1

---

## 清理空间

Docker 用久了会堆积大量无用镜像和容器，占用磁盘。

In [ ]:
# 查看 Docker 占用的磁盘空间
! docker system df

# 一键清理（删除停止的容器、未使用的网络、悬空镜像、构建缓存）
# ! docker system prune -a
# 加上 -a 会删除所有未使用的镜像（不只是悬空镜像）
# 慎用！先确认不需要再执行

---

## 🎯 第1周总结

| 概念 | 一句话 |
|------|--------|
| **Docker 本质** | 把你的应用和环境打包成标准化的容器 |
| **镜像 vs 容器** | 镜像 = 模具（只读），容器 = 饼干（可运行） |
| **docker run** | 创建并启动容器的核心命令 |
| **-p** | 端口映射，让外面能访问容器 |
| **-v** | 卷挂载，让容器读写宿主机文件 |
| **-e** | 传环境变量 |
| **--network** | 让容器之间通过容器名互联 |
| **docker exec -it ... bash** | 进入容器调试 |

### 核心命令肌肉记忆

```bash
docker ps                  # 看谁在跑
docker ps -a               # 看全部（包括死的）
docker logs <name>         # 看日志
docker exec -it <name> sh  # 进容器
docker stop <name>         # 关容器
docker rm <name>           # 删容器
docker rm -f <name>        # 强行删
docker system prune        # 大扫除
```

---

## 🧪 综合练习：手动搭建 Python + Redis + MySQL 环境

用本周学到的命令，在 Docker 中启动以下三个容器，并验证它们之间可以通信：

1. **Python 3.12** 容器（挂载当前目录 `/tmp/py-code`）
2. **Redis 7** 容器（做缓存）
3. **MySQL 8** 容器（root 密码 `mypass`）

要求：
- 三个容器在同一个自定义网络中（`practice-net`）
- Python 容器能 `ping` 通 redis 和 mysql（用容器名）
- MySQL 的数据目录挂载到宿主机 `/tmp/mysql-data`

> 提示：Python slim 镜像没有 ping 命令，需要先 `apt-get update && apt-get install -y iputils-ping` 或者用 Python 的 socket 模块验证

In [ ]:
# 你的练习代码写在这里

# Step 1: 创建网络
# ! docker network create practice-net

# Step 2: 启动 MySQL
# ! docker run -d --name practice-mysql \
#     --network practice-net \
#     -e MYSQL_ROOT_PASSWORD=mypass \
#     -v /tmp/mysql-data:/var/lib/mysql \
#     mysql:8

# Step 3: 启动 Redis
# ...

# Step 4: 启动 Python 并测试连接
# ...

pass